# 图解DeepSeek-R1

DeepSeek-R1的发布如同人工智能进步长卷中新迸发的最强音。对于机器学习研发社区而言，这个版本具有里程碑意义，主要原因包括：

· 它是一个公开权重的模型，并提供经过蒸馏的精简版本；

· 它公开了如何复现类似OpenAI o1的推理模型的训练方法，并公开了训练过程中的一些反思。

本附录将深入解析该模型的构建过程。

## A.1 回顾：大模型的训练原理


与现有大多数大模型相同，DeepSeek-R1的训练采用了逐词元生成的范式。但DeepSeek-R1在处理数学与推理问题上表现尤为出色，这种能力源于特殊的训练方法——它能够生成可以解释其思维链的“思考词元”(thinking token)，这样就能花更多时间来深入处理问题，如图A-1所示。

图A-1：思考词元

<div align="center">
  <img src="./picture/A-1.jpg" width="600">
  <p>图A-1：思考词元</p>
</div>

如图A-2展示了构建高质量大模型的通用方案，具体流程主要包含三个关键阶段。

图A-2：创建高质量LLM的三个关键阶

<div align="center">
  <img src="./picture/A-2.jpg" width="600">
  <p>图A-2：创建高质量LLM的三个关键阶段</p>
</div>

· 语言建模阶段：通过海量网络数据训练模型预测下一个词元，该阶段产出基座模型。

· 监督微调阶段：使模型在遵循指令和回答问题方面更加实用，该阶段产出指令调优模型或监督微调模型。

· 偏好调优阶段：最终通过人类偏好对齐进一步优化模型行为，生成可供应用程序交互的偏好调优大模型。

## A.2 DeepSeek-R1训练方案


DeepSeek-R1遵循这一通用方案。其中第一阶段的具体实施细节源自DeepSeek-V3模型的相关论文“DeepSeek-V3Technical Report”​。DeepSeek-R1使用该论文中的基座模型（而非最终的DeepSeek-V3模型）​，仍然经过监督微调和偏好调整阶段，但其具体实施方法存在创新性差异，如图A-3所示。

图A-3：DeepSeek-R1的训练过程

<div align="center">
  <img src="./picture/A-3.jpg" width="600">
  <p>图A-3：DeepSeek-R1的训练过程</p>
</div>

请注意，在DeepSeek-R1的创建过程中，有三个特别之处，接下来我们分别聊一聊。

### A.2.1 长推理链监督微调数据

基于基座模型进行监督微调需要用到数量庞大的长链思维推理示例（DeepSeek-R1的训练过程一共用到了60万个）​，如图A-4所示。如此规模的标注工作意味着，一方面数据极难获取，另一方面人工标注成本极为高昂。这也正是生成这些数据的过程成为值得强调的创新点的原因。

图A-4：推理数据：长链思维示例

<div align="center">
  <img src="./picture/A-4.jpg" width="600">
  <p>图A-4：推理数据：长链思维示例</p>
</div>

A.2.2 临时性高质量推理大模型

上述长链思维推理数据由DeepSeek-R1模型的前身生成，这是一个未命名的临时模型，专精于推理任务，其设计灵感源于另一个名为DeepSeek-R1-Zero的模型（稍后我们将详细讨论）​。这个临时模型的重大意义不在于它作为一个实用的大模型具备多好的性能，而在于其构建过程仅需少量标注数据，配合大规模强化学习训练，最终造就了一个擅长解决推理问题的模型。

通过使用这个未命名的专精于推理（在非推理任务上表现欠佳）的临时模型生成输出结果，开发者得以训练出更通用的模型，如图A-5所示。最终得到的通用模型不仅能完成推理任务，同时，在非推理任务的处理水平上，也能达到用户对大模型的预期。

图A-5：专精于推理的临时推理模型

<div align="center">
  <img src="./picture/A-5.jpg" width="600">
  <p>图A-5：专精于推理的临时推理模型</p>
</div>

### A.2.3 利用大规模强化学习构建推理模型

构建临时推理模型的关键就在于大规模推理导向的强化学习，如图A-6所示。

图A-6：推理导向的强化学习

<div align="center">
  <img src="./picture/A-6.jpg" width="600">
  <p>图A-6：推理导向的强化学习</p>
</div>

具体分为两个步骤：

· 大规模推理导向的强化学习(DeepSeek-R1-Zero)；

· 使用临时推理模型构建监督微调推理数据。

在DeepSeek-R1的训练方法中，强化学习用于构建临时推理模型。该模型随后用于生成推理示例，进一步用于监督微调。但实现这一模型的关键在于研发团队之前进行的一项实验，即训练DeepSeek-R1-Zero初始模型的实验，如图A-7所示。

图A-7：DeepSeek-R1-Zero和OpenAI o1在推理相关基准测试上的表现对比

<div align="center">
  <img src="./picture/A-7.jpg" width="600">
  <p>图A-7：DeepSeek-R1-Zero和OpenAI o1在推理相关基准测试上的表现对比</p>
</div>

该截图中的表来自论文“DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning”​，是DeepSeek-R1-Zero和OpenAI o1在推理相关基准测试上的表现对比。我们在后续内容中还会引用这篇论文中的内容，不妨将其简称为“DeepSeek-R1论文”​。

DeepSeek-R1-Zero的特殊性在于，它无须依赖标注的监督微调训练集，就能在推理任务中表现得很出色。其训练过程直接从预训练的基座模型开始，通过强化学习训练流程完成（跳过了监督微调阶段）​，如图A-8所示。它的表现非常出色，甚至能够与o1媲美。

图A-8：DeepSeek-R1-Zero的训练：只有推理导向的强化学习

<div align="center">
  <img src="./picture/A-8.jpg" width="600">
  <p>图A-8：DeepSeek-R1-Zero的训练：只有推理导向的强化学习</p>
</div>

这一点之所以重要，是因为数据始终是机器学习模型能力的关键所在。DeepSeek-R1-Zero如何能突破这一历史定式？这指向两个关键事实。

· 现代基座模型已经跨越了某个门槛，在质量和能力上有了质的提升（DeepSeek-V3-Base基于14.8万亿个高质量词元进行训练）​。

·与通用对话或写作需求不同，推理问题可以实现自动验证与标注。让我们通过具体示例加以说明。

· 示例：推理问题的自动验证

在强化学习训练阶段，假设给出如下提示词（或者输入如下问题）​：

编写一段python代码：要求输入一个数字列表，将其排序，之后再列表的开头添加42并返回最终结果。

此类问题天然适合多种自动验证方式。假设我们将该问题输入正在训练的模型，并得到一个输出，以下多种方法都可以对输出做自动验证：

· 通过代码检查工具（比如linter）验证输出是否为合法的Python代码；

· 执行生成的Python代码，以检验其运行状态；

· 借助其他现代编程大模型，创建单元测试来验证预期功能（这类模型自身无须具备推理能力）​；

· 我们甚至可以进一步测量执行时间，在训练过程中优先选择性能更好的解决方案——即使其他方案同样能解决问题。

在训练阶段，我们可以向模型提出此类问题，并生成多个可能的解决方案，如图A-9所示。

图A-9：在训练阶段生成多个解决方案

<div align="center">
  <img src="./picture/A-9.jpg" width="600">
  <p>图A-9：在训练阶段生成多个解决方案</p>
</div>

通过上述自动验证（无须人工干预）​，我们可以发现：

· 第一个结果甚至不是有效的代码；

· 第二个结果虽然是代码，但并不是Python语言编写的；

· 第三个结果看似可行，但没有通过单元测试；

· 第四个结果才是完全正确的解决方案。

这个自动验证的过程如图A-10所示。

图A-10：在训练阶段进行自动验证

<div align="center">
  <img src="./picture/A-10.jpg" width="600">
  <p>图A-10：在训练阶段进行自动验证</p>
</div>

这类反馈信号都可以直接用于模型优化，如图A-11所示。训练过程自然需要基于大量的示例（以小批量形式）持续迭代，并通过反复的训练逐步实现。

图A-11：反馈信号用于优化模型

<div align="center">
  <img src="./picture/A-11.jpg" width="600">
  <p>图A-11：反馈信号用于优化模型</p>
</div>

这些奖励信号与模型参数更新机制，正是驱动模型在强化学习训练过程中持续提升任务表现的核心原理——该原理在DeepSeek-R1论文中亦有直观呈现，如图A-12所示。

图A-12：反馈信号用于持续优化DeepSeek-R1-Zero

<div align="center">
  <img src="./picture/A-12.jpg" width="600">
  <p>图A-12：反馈信号用于持续优化DeepSeek-R1-Zero</p>
</div>

该注释图来自DeepSeek-R1论文。对于每个问题，DeepSeek研发团队随机抽取16个响应并计算整体平均准确率以确保评估的稳定性。

与这种能力提升相呼应的是生成响应的长度——模型会通过生成更多思考词元来处理复杂问题，如图A-13所示。

图A-13：DeepSeek-R1-Zero在训练集上的平均响应长度

<div align="center">
  <img src="./picture/A-13.jpg" width="600">
  <p>图A-13：DeepSeek-R1-Zero在训练集上的平均响应长度</p>
</div>

该注释图来自DeepSeek-R1论文。图中展示的是在强化学习训练期间，DeepSeek-R1-Zero在训练集上的平均响应长度。DeepSeek-R1-Zero自然而然地学会了通过更长时间的思考来解决推理任务。

虽然这一流程行之有效，DeepSeek-R1-Zero模型在这些推理问题上得分也颇高，但它仍面临其他一些问题，导致其实际可用性不及预期。

尽管DeepSeek-R1-Zero展现出了出色的推理能力，并能自主发展出某些预料之外的强大的推理行为，但它也面临不少挑战，例如，生成的内容可读性差、语言混杂等。

DeepSeek-R1的目标就是成为更具实用价值的模型。因此，研发团队并未完全依赖强化学习流程，而是如前所述，在以下两个环节（如图A-14所示）应用强化学习：

· 创建一个临时推理模型来生SFT数据点；

· 训练DeepSeek-R1模型以提升其在推理类和非推理类问题上的表现（使用其他类型的验证器）​。

图A-14：应用强化学习的两个关键环节：(1)和(2)

<div align="center">
  <img src="./picture/A-14.jpg" width="600">
  <p>图A-14：应用强化学习的两个关键环节：(1)和(2)</p>
</div>

· 使用临时推理模型构建监督微调推理数据

为使临时推理模型更具实用性，需基于数千个推理问题示例（部分由DeepSeek-R1-Zero生成并经过了筛选）进行监督微调训练。研发团队将其称为“冷启动数据”(cold startdata)，如图A-15所示。

图A-15：冷启动数据

<div align="center">
  <img src="./picture/A-15.jpg" width="600">
  <p>图A-15：冷启动数据</p>
</div>

#### 冷启动

与DeepSeek-R1-Zero不同，为了避免基座模型在强化学习训练初期出现不稳定的冷启动阶段，DeepSeek-R1采取了相应的措施。我们构建并收集了少量长链思维数据，对模型进行微调并将其作为强化学习训练的初始演员(actor)模型。为收集此类数据，我们探索了多种方法：使用包含长链思维示例的少样本提示词，直接通过提示词引导模型生成包含反思与验证的详细答案，将DeepSeek-R1-Zero生成的输出内容整理成易于阅读的格式，以及结合人工标注对结果进行后处理，进一步优化数据质量。

以上内容来自DeepSeek-R1论文。

不过稍等，既然我们已经有了这些数据，为何还要依赖强化学习过程？原因就在于数据的规模。这个冷启动数据集可能仅有5000个示例（这在实际操作中确实不难获取）​，但要训练DeepSeek-R1模型却需要60万个示例。这一临时模型的关键作用在于弥合数据量的鸿沟，使我们能够通过自动合成的方式生成那些极具训练价值的数据，如图A-16所示。

图A-16：临时模型的关键作用

<div align="center">
  <img src="./picture/A-16.jpg" width="600">
  <p>图A-16：临时模型的关键作用</p>
</div>

如果你对监督微调的概念还不熟悉，这一过程的本质是向模型提供由提示词和正确的补全结果）构成的训练示例。图A-17展示了几个SFT训练示例。

图A-17：监督微调训练示例：用户提供的指令数据及模型的回答

<div align="center">
  <img src="./picture/A-17.jpg" width="600">
  <p>图A-17：监督微调训练示例：用户提供的指令数据及模型的回答</p>
</div>

· 通用强化学习训练阶段

这使得DeepSeek-R1不仅在推理任务上表现出色，也能胜任非推理任务。该流程与我们之前介绍的强化学习过程类似，但由于其应用范围扩展到了非推理领域，因此会针对属于这些应用场景的提示词，采用有用性奖励模型和安全性奖励模型（类似于Llama模型的机制）进行训练，如图A-18所示。

图A-18：通用强化学习训练流程

<div align="center">
  <img src="./picture/A-18.jpg" width="600">
  <p>图A-18：通用强化学习训练流程</p>
</div>